# Synthetic Dataset Generator — Version 1

This is Version 1 of my Synthetic Dataset Generator project, built as part of Week 3 of the AI Engineering Core Track.

I worked through this project step by step and understood the architecture, the libraries, the provider-specific generation routes, the validation flow, and the Gradio interface before having Codex help implement the code. I did not write every line myself, but I understand how the application works and made implementation decisions as we built it.

The project uses a deliberately simple application schema, for example `{"name": "string", "age": "integer"}`. This is not formal JSON Schema. Formal JSON Schema appears only where OpenAI Structured Outputs requires it.


## 1. Setup & Runtime

Install once in a fresh Colab runtime, then import the packages. The original recorded Colab environment was a Tesla T4 with PyTorch 2.11.0+cu128, Transformers 5.15.1, bitsandbytes 0.50.1, Accelerate 1.14.0, Gradio 6.24.0, and OpenAI 3.3.1. Those are historical observations, not pinned requirements.


In [ ]:
# Run once in a fresh Google Colab runtime.
!pip install -q -U transformers accelerate bitsandbytes gradio openai pandas


In [ ]:
import gc
import json
import os
import time

import accelerate
import bitsandbytes as bnb
import gradio
import openai
import pandas as pd
import torch
import transformers
from openai import OpenAI
from pandas.api.types import is_bool_dtype, is_integer_dtype, is_numeric_dtype, is_string_dtype
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"PyTorch version      : {torch.__version__}")
print(f"Transformers version : {transformers.__version__}")
print(f"Accelerate version   : {accelerate.__version__}")
print(f"BitsAndBytes version : {bnb.__version__}")
print(f"Gradio version       : {gradio.__version__}")
print(f"OpenAI version       : {openai.__version__}")
print(f"CUDA available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                  : {torch.cuda.get_device_name(0)}")
else:
    print("GPU-only model cells are preserved below but should be skipped in this runtime.")


## 2. Hugging Face Fundamentals

The first model is the small, ungated `Qwen/Qwen2.5-1.5B-Instruct`. The following cells intentionally show the manual route from ordinary text to token IDs, tensors, `model.generate()`, and decoded text before using a reusable helper.


In [ ]:
SMALL_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(SMALL_MODEL_NAME)
print(type(tokenizer))


In [ ]:
text = "Hello, how are you?"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Tokens:", tokens)
print("Token IDs:", token_ids)
print("Decoded again:", tokenizer.decode(token_ids))


**Recorded original tokenization:** `['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', '?']` became `[9707, 11, 1246, 525, 498, 30]`. Decoding those IDs returned the original sentence. The `Ġ` marker is part of this tokenizer's representation of a preceding space.


In [ ]:
# This download/load needs a suitable GPU in Colab.
model = AutoModelForCausalLM.from_pretrained(
    SMALL_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)
print(type(model))
print(f"Memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GiB")
print(f"Reported dtype: {model.dtype}")


In [ ]:
# Manual inference: text -> tensors -> generate() -> generated token IDs -> text.
prompt = "What is the capital of France?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output_ids = model.generate(**inputs, max_new_tokens=50)

input_length = inputs["input_ids"].shape[1]
generated_token_ids = output_ids[0][input_length:]
response = tokenizer.decode(generated_token_ids, skip_special_tokens=True)

print("Input tensor shape:", inputs["input_ids"].shape)
print("Generated answer:", response)


In [ ]:
# A chat template adds the model-specific message markers.
messages = [{"role": "user", "content": "What is the capital of France?"}]
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print(formatted_prompt)


In [ ]:
def generate_text_local(
    prompt,
    model,
    tokenizer,
    max_new_tokens=600,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    enable_thinking=None,
):
    '''Generate only the assistant's new text from a local chat model.'''
    messages = [{"role": "user", "content": prompt}]
    template_kwargs = {
        "tokenize": True,
        "add_generation_prompt": True,
        "return_tensors": "pt",
        "return_dict": True,
    }
    # Qwen3.5 understands this template option. Qwen2.5 does not need it.
    if enable_thinking is not None:
        template_kwargs["enable_thinking"] = enable_thinking

    model_inputs = tokenizer.apply_chat_template(messages, **template_kwargs)
    model_inputs = model_inputs.to(model.device)

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
    }
    if do_sample:
        generation_kwargs.update({"temperature": temperature, "top_p": top_p})

    output_ids = model.generate(**model_inputs, **generation_kwargs)
    input_length = model_inputs["input_ids"].shape[1]
    new_token_ids = output_ids[0][input_length:]
    return tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()


In [ ]:
response = generate_text_local(
    "Give me three fictional customer names.",
    model=model,
    tokenizer=tokenizer,
)
print(response)


## 3. From LLM Text to Structured Data

“Return valid JSON” is still an instruction to a language model, not a guarantee that `json.loads()` can immediately parse the response. The original run returned a JSON array inside Markdown fences. Keep the useful cleanup step, then parse records and make a DataFrame.

**Recorded original result:** Qwen returned three customer records as a fenced `json` block. After removing the fences, `json.loads()` returned a Python `list` of `dict` objects and `pd.DataFrame()` displayed the three rows.


In [ ]:
def strip_json_code_fence(text):
    '''Remove one optional Markdown code fence around a JSON response.'''
    cleaned = text.strip()
    if cleaned.startswith("```"):
        first_newline = cleaned.find("\n")
        if first_newline == -1:
            raise ValueError("The response contains an incomplete Markdown code fence.")
        cleaned = cleaned[first_newline + 1:]
        if cleaned.rstrip().endswith("```"):
            cleaned = cleaned.rstrip()[:-3]
    return cleaned.strip()


def parse_json_records(response_text):
    cleaned = strip_json_code_fence(response_text)
    records = json.loads(cleaned)
    if not isinstance(records, list) or not all(isinstance(row, dict) for row in records):
        raise ValueError("Expected a JSON array of record objects.")
    return records


In [ ]:
json_prompt = ''' 
Generate 3 fictional customer records with name, age, and city.
Return only a JSON array. Do not add an explanation.
'''
response = generate_text_local(json_prompt, model=model, tokenizer=tokenizer)
records = parse_json_records(response)
customer_df = pd.DataFrame(records)
customer_df


**Recorded original parsed data:** `[{'name': 'Alice Johnson', 'age': 28, 'city': 'New York'}, {'name': 'Bob Lee', 'age': 45, 'city': 'Los Angeles'}, {'name': 'Charlie Brown', 'age': 17, 'city': 'Chicago'}]`. The resulting DataFrame had one row for each record.


## 4. General-Purpose Dataset Generation & Validation

The application-facing schema stays simple. Supported values are `string`, `integer`, `number`, and `boolean`. Here `number` intentionally accepts either an integer or a decimal number.


In [ ]:
SUPPORTED_TYPES = {"string", "integer", "number", "boolean"}


def validate_simple_schema(schema):
    '''Check the small schema format the future app will accept.'''
    if not isinstance(schema, dict) or not schema:
        raise ValueError("Schema must be a non-empty dictionary of column names and supported types.")
    for column, expected_type in schema.items():
        if not isinstance(column, str) or not column.strip():
            raise ValueError("Every schema field name must be a non-empty string.")
        if expected_type not in SUPPORTED_TYPES:
            raise ValueError(
                f"Unsupported type for '{column}': {expected_type}. "
                f"Choose from {sorted(SUPPORTED_TYPES)}."
            )


def build_local_dataset_prompt(dataset_description, schema, num_rows):
    validate_simple_schema(schema)
    if not isinstance(num_rows, int) or num_rows < 1:
        raise ValueError("num_rows must be a positive integer.")
    return f'''Generate exactly {num_rows} fictional records for this dataset:

{dataset_description}

Each record must follow this simple schema:
{json.dumps(schema, indent=2)}

Return only a JSON array. Do not use Markdown code fences or add any explanation.'''


In [ ]:
# This is the original pandas-dtype lesson. It is useful for inspecting a DataFrame,
# but the canonical validator below also checks each value.
def validate_dataframe_dtypes(df, schema):
    for column, expected_type in schema.items():
        actual_dtype = df[column].dtype
        if expected_type == "string":
            valid = is_string_dtype(actual_dtype)
        elif expected_type == "integer":
            valid = is_integer_dtype(actual_dtype)
        elif expected_type == "number":
            valid = is_numeric_dtype(actual_dtype) and not is_bool_dtype(actual_dtype)
        else:  # boolean
            valid = is_bool_dtype(actual_dtype)
        if not valid:
            raise ValueError(f"Column '{column}' should be {expected_type}, but got {actual_dtype}.")


In [ ]:
def value_matches_type(value, expected_type):
    # bool is checked before integer because Python treats bool as a subclass of int.
    if expected_type == "string":
        return isinstance(value, str)
    if expected_type == "integer":
        return isinstance(value, int) and not isinstance(value, bool)
    if expected_type == "number":
        return isinstance(value, (int, float)) and not isinstance(value, bool)
    return isinstance(value, bool)


def validate_records(records, schema, num_rows):
    '''Validate count, exact columns, and every value before creating a DataFrame.'''
    validate_simple_schema(schema)
    if len(records) != num_rows:
        raise ValueError(f"Expected {num_rows} rows, but model returned {len(records)}.")

    expected_columns = set(schema)
    for row_number, record in enumerate(records, start=1):
        if set(record) != expected_columns:
            raise ValueError(
                f"Row {row_number} has columns {sorted(record)}; "
                f"expected {sorted(expected_columns)}."
            )
        for column, expected_type in schema.items():
            if not value_matches_type(record[column], expected_type):
                raise ValueError(
                    f"Row {row_number}, column '{column}' should be {expected_type}; "
                    f"got {record[column]!r}."
                )


In [ ]:
def generate_dataset_basic(
    dataset_description, schema, num_rows, model, tokenizer, enable_thinking=None
):
    '''Local-model route: prompt -> JSON text -> records -> validated DataFrame.'''
    prompt = build_local_dataset_prompt(dataset_description, schema, num_rows)
    response_text = generate_text_local(
        prompt,
        model=model,
        tokenizer=tokenizer,
        enable_thinking=enable_thinking,
    )
    records = parse_json_records(response_text)
    validate_records(records, schema, num_rows)
    return pd.DataFrame(records)


product_schema = {
    "product_name": "string",
    "category": "string",
    "price": "number",
    "rating": "number",
    "in_stock": "boolean",
}

product_df = generate_dataset_basic(
    dataset_description="Products sold by an Indian electronics retailer",
    schema=product_schema,
    num_rows=5,
    model=model,
    tokenizer=tokenizer,
)
validate_dataframe_dtypes(product_df, product_schema)
product_df


**Recorded original dtype result:** the five product fields were inferred as `object`, `object`, `int64`, `float64`, and `bool`. The dtype helper therefore accepted both the integer price and floating-point rating as `number` fields.


## 5. Sampling & Diversity Experiments

Sampling makes token selection probabilistic. `temperature` changes how sharply probabilities are weighted; `top_p` limits selection to the smallest token set whose cumulative probability reaches that value. The original notebook tested temperatures 0.3, 0.7, and 1.2, plus `top_p` values 0.5, 0.9, and 1.0.

**Recorded takeaway:** a practical baseline for this small local model was `do_sample=True`, `temperature=0.7`, and `top_p=0.8` or `0.9`. These were exploratory examples, not a benchmark.


In [ ]:
def run_sampling_experiment(dataset_description, schema, model, tokenizer, temperature, top_p):
    prompt = build_local_dataset_prompt(dataset_description, schema, num_rows=5)
    response_text = generate_text_local(
        prompt,
        model=model,
        tokenizer=tokenizer,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
    )
    records = parse_json_records(response_text)
    validate_records(records, schema, num_rows=5)
    return pd.DataFrame(records)


# Run one concise experiment. Change only the named parameters to compare results.
sampled_products = run_sampling_experiment(
    "Products sold by an American clothing brand",
    product_schema,
    model=model,
    tokenizer=tokenizer,
    temperature=0.7,
    top_p=0.9,
)
sampled_products


## 6. Quantization & Local-Model Comparison

The original Colab run reported about **2.88 GiB** for Qwen2.5-1.5B in BF16 and **2.85 GiB** for Qwen3.5-4B loaded in 4-bit mode. This demonstrates that 4-bit quantization mainly reduces weight storage; it does not mean every tensor or calculation is 4-bit. The quantized model still reported `torch.bfloat16` for its compute/non-quantized parts.

Qwen3.5 initially spent output on thinking. `enable_thinking=False` is retained below to request direct answers for this structured-data experiment. The larger model is optional and is not required by the final project.


In [ ]:
# The first model is no longer needed. Delete the reference before clearing the CUDA cache.
del model
del tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released the Qwen2.5-1.5B model before the optional 4-bit comparison.")


In [ ]:
RUN_LARGE_MODEL_EXPERIMENT = False  # Set to True only when you want to repeat this optional comparison.

if RUN_LARGE_MODEL_EXPERIMENT:
    LARGE_MODEL_NAME = "Qwen/Qwen3.5-4B"
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    large_tokenizer = AutoTokenizer.from_pretrained(LARGE_MODEL_NAME)
    large_model = AutoModelForCausalLM.from_pretrained(
        LARGE_MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
    )
    print(f"4-bit model footprint: {large_model.get_memory_footprint() / 1024**3:.2f} GiB")
    print(f"Reported dtype: {large_model.dtype}")
else:
    large_model = None
    large_tokenizer = None
    print("Skipped optional Qwen3.5-4B comparison. Set RUN_LARGE_MODEL_EXPERIMENT=True to run it.")


In [ ]:
if large_model is None:
    print("Skipped optional Qwen3.5-4B timing test.")
else:
    start = time.perf_counter()
    product_df_4b = generate_dataset_basic(
        dataset_description="Products sold by an American clothing brand",
        schema=product_schema,
        num_rows=5,
        model=large_model,
        tokenizer=large_tokenizer,
        enable_thinking=False,
    )
    elapsed_4b = time.perf_counter() - start
    print(f"Generation time: {elapsed_4b:.2f} seconds")
    print(product_df_4b)


**Recorded original timing:** Qwen3.5-4B 4-bit generated the five clothing records in about **27.36 seconds**. It produced valid data in that run. This was one observed run, not a scientific speed or quality benchmark.


In [ ]:
if large_model is not None:
    del large_model
    del large_tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Reload the smaller model only for the like-for-like timing comparison.
small_tokenizer = AutoTokenizer.from_pretrained(SMALL_MODEL_NAME)
small_model = AutoModelForCausalLM.from_pretrained(
    SMALL_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)


In [ ]:
start = time.perf_counter()
product_df_1_5b = generate_dataset_basic(
    dataset_description="Products sold by an American clothing brand",
    schema=product_schema,
    num_rows=5,
    model=small_model,
    tokenizer=small_tokenizer,
)
elapsed_1_5b = time.perf_counter() - start
print(f"Generation time: {elapsed_1_5b:.2f} seconds")
product_df_1_5b


**Recorded original timing:** Qwen2.5-1.5B BF16 generated the same size dataset in about **22.45 seconds** and passed validation. A model fitting into GPU memory is not automatically the fastest or most practical choice.


In [ ]:
del small_model
del small_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 7. OpenAI API & Structured Outputs

OpenAI uses the Responses API here. The API key is read from Colab Secrets when available, then falls back to an environment variable for other environments. It is never written into the notebook.

**Recorded original results:** the connection check returned `OpenAI connection works`; a plain-JSON five-product call took about **6.90 seconds**. This is one environment-specific observation, not evidence that hosted APIs are universally faster than local models.


In [ ]:
def get_openai_api_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENAI_API_KEY")
    except ImportError:
        return os.getenv("OPENAI_API_KEY")


OPENAI_API_KEY = get_openai_api_key()
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
print("OpenAI client is ready." if openai_client else "OpenAI calls will be skipped: no API key found.")


In [ ]:
# Tiny, optional connection check. It does not run when no key is configured.
if openai_client is None:
    print("Skipped OpenAI connection check.")
else:
    response = openai_client.responses.create(
        model="gpt-5-nano",
        input="Reply with exactly: OpenAI connection works",
    )
    print(response.output_text)


### Simple schema versus formal JSON Schema

The app's simple schema says what columns the user wants. JSON Schema is a separate, formal set of rules for data. OpenAI Structured Outputs is given the formal rules; the model returns **data** that follows them. The response uses an object containing `records` because the original experiment found that an array root was rejected.


In [ ]:
JSON_SCHEMA_TYPES = {
    "string": "string",
    "integer": "integer",
    "number": "number",
    "boolean": "boolean",
}


def simple_schema_to_openai_schema(schema, num_rows, records_key="records"):
    '''Convert the app's small schema into the JSON Schema used by Structured Outputs.'''
    validate_simple_schema(schema)
    if not isinstance(num_rows, int) or num_rows < 1:
        raise ValueError("num_rows must be a positive integer.")

    record_properties = {
        column: {"type": JSON_SCHEMA_TYPES[expected_type]}
        for column, expected_type in schema.items()
    }
    record_schema = {
        "type": "object",
        "properties": record_properties,
        "required": list(schema),
        "additionalProperties": False,
    }
    return {
        "type": "object",
        "properties": {
            records_key: {
                "type": "array",
                "items": record_schema,
                "minItems": num_rows,
                "maxItems": num_rows,
            }
        },
        "required": [records_key],
        "additionalProperties": False,
    }


In [ ]:
def generate_records_openai(dataset_description, schema, num_rows, client, model_name="gpt-5-nano"):
    '''OpenAI route: simple schema -> formal schema -> structured JSON -> records.'''
    if client is None:
        raise RuntimeError("An OpenAI API key is required for the OpenAI provider.")

    records_key = "records"
    response = client.responses.create(
        model=model_name,
        input=(
            f"Generate realistic, fictional records for this dataset: {dataset_description}. "
            "Return exactly the requested records."
        ),
        text={
            "format": {
                "type": "json_schema",
                "name": "synthetic_dataset",
                "strict": True,
                "schema": simple_schema_to_openai_schema(schema, num_rows, records_key),
            }
        },
    )
    payload = json.loads(response.output_text)
    records = payload[records_key]
    validate_records(records, schema, num_rows)
    return records


if openai_client is None:
    print("Skipped tiny Structured Outputs test.")
else:
    structured_df = pd.DataFrame(
        generate_records_openai(
            "Products sold by an American clothing brand",
            product_schema,
            num_rows=5,
            client=openai_client,
        )
    )
    structured_df


## 8. Final Synthetic Dataset Generator

**Above:** learning and experiments. **From here:** one canonical, function-based implementation. It accepts the app's simple schema and selects a provider without hiding the important steps behind classes or framework machinery.


In [ ]:
def generate_records_huggingface(
    dataset_description, schema, num_rows, model, tokenizer, enable_thinking=None
):
    prompt = build_local_dataset_prompt(dataset_description, schema, num_rows)
    response_text = generate_text_local(
        prompt,
        model=model,
        tokenizer=tokenizer,
        enable_thinking=enable_thinking,
    )
    records = parse_json_records(response_text)
    validate_records(records, schema, num_rows)
    return records


def generate_dataset(
    dataset_description,
    schema,
    num_rows,
    provider,
    *,
    local_model=None,
    local_tokenizer=None,
    local_enable_thinking=None,
    openai_client=None,
    openai_model="gpt-5-nano",
):
    '''Return one validated pandas DataFrame from the selected provider.'''
    validate_simple_schema(schema)

    if provider == "huggingface":
        if local_model is None or local_tokenizer is None:
            raise ValueError("The Hugging Face provider needs a loaded model and tokenizer.")
        records = generate_records_huggingface(
            dataset_description,
            schema,
            num_rows,
            local_model,
            local_tokenizer,
            local_enable_thinking,
        )
    elif provider == "openai":
        records = generate_records_openai(
            dataset_description, schema, num_rows, openai_client, openai_model
        )
    else:
        raise ValueError("provider must be 'huggingface' or 'openai'.")

    return pd.DataFrame(records)


In [ ]:
# Canonical OpenAI example. It stays safely skipped when no key is present.
customer_schema = {
    "name": "string",
    "age": "integer",
    "city": "string",
    "membership_type": "string",
    "annual_spend": "number",
    "active": "boolean",
}

if openai_client is None:
    print("Canonical OpenAI example skipped: add OPENAI_API_KEY in Colab Secrets to run it.")
else:
    customer_df = generate_dataset(
        dataset_description="Customers of an Indian e-commerce company",
        schema=customer_schema,
        num_rows=5,
        provider="openai",
        openai_client=openai_client,
    )
    customer_df


## 9. Gradio App

### Step 1 — UI skeleton

This first step creates the visible interface only. The button is intentionally not connected to `generate_dataset()` yet.


In [ ]:
import gradio as gr


def generate_dataset_from_ui(dataset_description, schema_text, num_rows, provider):
    schema = json.loads(schema_text)
    return generate_dataset(
        dataset_description=dataset_description,
        schema=schema,
        num_rows=int(num_rows),
        provider=provider,
        local_model=globals().get("model"),
        local_tokenizer=globals().get("tokenizer"),
        openai_client=globals().get("openai_client"),
    )


with gr.Blocks(title="Synthetic Dataset Generator") as demo:
    gr.Markdown(
        "# Synthetic Dataset Generator\n"
        "Describe the fictional dataset you want, define its simple schema, "
        "and choose a provider."
    )

    dataset_description = gr.Textbox(
        label="Dataset description",
        placeholder="Example: Customers of an Indian e-commerce company",
        lines=3,
    )
    schema_text = gr.Code(
        label="Simple schema (JSON)",
        language="json",
        value=json.dumps(customer_schema, indent=2),
        lines=8,
    )
    num_rows = gr.Slider(
        minimum=1,
        maximum=100,
        value=5,
        step=1,
        precision=0,
        label="Number of rows",
    )
    provider = gr.Dropdown(
        choices=["huggingface", "openai"],
        value="openai",
        label="Generation provider",
    )
    generate_button = gr.Button("Generate Dataset")
    generated_dataset = gr.Dataframe(
        value=pd.DataFrame(columns=customer_schema),
        label="Generated dataset",
        type="pandas",
        interactive=False,
    )

    generate_button.click(
        fn=generate_dataset_from_ui,
        inputs=[dataset_description, schema_text, num_rows, provider],
        outputs=generated_dataset,
    )


demo.launch()
